In [5]:
import csv
import logging
import os
import time
import re
from typing import Union
import numpy as np
import pandas as pd
from pathlib import Path
import json

from aind_codeocean_pipeline_monitor.models import (CaptureSettings,
                                                    PipelineMonitorSettings)
from aind_data_access_api.document_db import MetadataDbClient
from codeocean import CodeOcean
from codeocean.computation import (ComputationState, DataAssetsRunParam,
                                   RunParams)
from dataclasses_json import dataclass_json

from lamf_analysis.code_ocean import docdb_utils
from lamf_analysis.code_ocean import capsule_data_utils as cdu
from lamf_analysis.code_ocean import code_ocean_utils as cou

%load_ext autoreload
%autoreload 2



In [6]:
subject_ids = [755252, 767018, 767022, 783551, 785054, 782149, 788406, 790322, 800792, 800995, 804363, 804670] # all Slc32a1;Oi1 collected so far


In [15]:
sczdrift_df

,single_cell_zdrift_derived_name,location,derived_asset_id,process,derived_date,raw_name
0,multiplane-ophys_804670_2025-09-12_09-36-10_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/3...,353e97e8-4e81-4986-bae8-d2215231cac5,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-12_09-36-10_si...
1,multiplane-ophys_804670_2025-09-20_09-16-09_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/d...,d909e23b-db4a-40b6-944b-995fd0fad318,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-20_09-16-09_si...
2,multiplane-ophys_804670_2025-09-30_10-05-45_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/d...,daace940-4c4b-4d77-86bc-3a50671b98ab,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-30_10-05-45_si...
3,multiplane-ophys_804670_2025-09-25_09-49-47_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/c...,c8f5b0a8-49fb-4aaa-95ce-26057b428f6f,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-25_09-49-47_si...
4,multiplane-ophys_804670_2025-09-11_09-40-54_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/4...,42a002b3-1e5a-4f75-a8d9-f866c03ec288,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-11_09-40-54_si...
5,multiplane-ophys_804670_2025-10-02_10-10-21_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/e...,ec4074b4-ebfd-4acd-b0f6-648896234666,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-10-02_10-10-21_si...
6,multiplane-ophys_804670_2025-09-24_09-30-56_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/d...,d7656f6c-7bbc-4551-824d-bb79fe15ddd0,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-24_09-30-56_si...
7,multiplane-ophys_804670_2025-10-01_10-15-38_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/3...,3c9074e9-3a19-4a21-8089-3ae057223819,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-10-01_10-15-38_si...
8,multiplane-ophys_804670_2025-09-19_09-38-54_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/8...,83e2d118-2fbc-4c74-9063-16284911afe5,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-19_09-38-54_si...
9,multiplane-ophys_804670_2025-09-27_09-39-25_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/c...,c648d976-9824-49bc-b73d-f5fadd5e70c1,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-27_09-39-25_si...


In [ ]:
session_infos = docdb_utils.get_session_infos_from_docdb(subject_id, filter_test_data=True)
processed_infos = docdb_utils.get_processed_data_info(subject_id).sort_values('long_window').drop_duplicates(subset=['raw_name'])
merged_df = processed_infos.merge(session_infos, left_on='raw_name', right_on='raw_asset_name', how='left')

sczdrift_df = docdb_utils.get_derived_data_assets(subject_ids[-1], 'single-cell-zdrift')
sczdrift_df.rename(columns={'derived_name': 'single_cell_zdrift_derived_name',
                            'derived_asset_id': 'single_cell_zdrift_derived_asset_id'}, inplace=True)
merged_df = merged_df.merge(sczdrift_df[['raw_name',
                                         'single_cell_zdrift_derived_name',
                                         'single_cell_zdrift_derived_asset_id']],
                            on='raw_name', how='left')


In [35]:
def get_attach_df(subject_id):
    # including single-cell-zdrift
    session_infos = docdb_utils.get_session_infos_from_docdb(subject_id, filter_test_data=True)
    processed_infos = docdb_utils.get_processed_data_info(subject_id).sort_values('long_window').drop_duplicates(subset=['raw_name'])
    merged_df = processed_infos.merge(session_infos, left_on='raw_name', right_on='raw_asset_name', how='left')

    sczdrift_df = docdb_utils.get_derived_data_assets(subject_id, 'single-cell-zdrift-qc')
    sczdrift_df.rename(columns={'derived_name': 'single_cell_zdrift_derived_name',
                                'derived_asset_id': 'single_cell_zdrift_derived_asset_id'}, inplace=True)
    merged_df = merged_df.merge(sczdrift_df[['raw_name',
                                            'single_cell_zdrift_derived_name',
                                            'single_cell_zdrift_derived_asset_id']],
                                on='raw_name', how='inner')
    # attach_df = merged_df[merged_df.session_type.str.contains('OPHYS_')]
    return merged_df

In [37]:
subject_id = subject_ids[0]
merged_df = get_attach_df(subject_id)
attach_asset_ids = merged_df['raw_asset_id'].tolist() + merged_df['processed_asset_id'].tolist() + merged_df['single_cell_zdrift_derived_asset_id'].tolist()
cou.attach_assets(attach_asset_ids)
# processed_infos

asset_id: 7b1f90f0-6cbe-4ea7-b353-9d74c6237e0f - mount_state: unchanged
asset_id: 1b187fe7-13ac-4800-9dba-b6523f22765c - mount_state: unchanged
asset_id: f7451b41-32e8-43b3-a961-d30ab09ba789 - mount_state: unchanged
asset_id: 73d47ce2-1d56-48db-9f7b-d99aaf782d1d - mount_state: unchanged
asset_id: 6c25405f-83a5-4284-b911-892f9e63d852 - mount_state: unchanged
asset_id: d7728098-17a5-4a47-ad48-4eba2217e4ed - mount_state: unchanged
asset_id: bf4b79f3-8b09-4fb3-ab7e-9b4148753d36 - mount_state: unchanged
asset_id: 929181ae-0346-408d-b658-8d2dfb24dba4 - mount_state: unchanged
asset_id: 7f9672e5-9cf5-4907-9bc4-a514f44d4eaa - mount_state: unchanged
asset_id: 95ba6dcc-c32a-4812-a234-27ae729cc497 - mount_state: unchanged
asset_id: 1a0a5408-a111-4383-b805-4fb685fca696 - mount_state: unchanged
asset_id: 8595d519-f2e0-4662-9909-56dd38d215bf - mount_state: unchanged
asset_id: feee58c2-4f89-4b9c-8cb4-eb01c71e2661 - mount_state: unchanged
asset_id: ecaf863b-df78-4539-b0b7-35fe5584f037 - mount_state: un